# ARC-AGI-3 — Qwen3.8 27B public benchmark on Colab

Runs the same 25-game public evaluation as Kaggle kernel `takumuhata/arc3-duck-qwen3-8-27b` (local mean **4.97**) on a Colab GPU. Reuses the **exact unmodified kernel notebook** via papermill — it stays in sync with whatever is pushed to Kaggle.

## How to run
1. **Runtime → Change runtime type → A100 GPU** (required — needs ~30GB VRAM. H100 also works. T4/V100/L4 will NOT work).
2. Have `credentials.json` for Kaggle user **takumuhata** ready (API credentials for the account that owns the datasets).
3. Run cells top to bottom. Downloads are ~30GB; the benchmark itself takes **5–8h** on A100. Colab sessions cap at 24h — OK.
4. Final score prints in the last cell (`score.json` + `summary.txt` under `/content/working` — browse it in the left Files panel).

**Long-run tips**: Colab Pro's "Run in background" option (Runtime menu) keeps the session alive if you close the tab. A plain foreground run also survives a closed tab for a few hours, but background is safer for a 5-8h benchmark. If the session dies before finishing, just re-run — downloads reuse `/content` only within the same session, so a dead session means re-downloading (~25 min).

## What it does
- Installs vLLM 0.19 + the private FP8 model `foysalemonshanto/qwen3-8-27b-fp8-repacked-v1` (served as `Qwen/Qwen3.8-27B-FP8`)
- Recreates the `/kaggle` mount layout the kernel expects (symlinks only)
- Executes the pulled kernel notebook verbatim, then reports the mean score

In [ ]:
# Cell 2 — GPU check (must be A100 or H100)
import subprocess, sys
out = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],capture_output=True,text=True)
print(out.stdout)
name = out.stdout.split(",")[0].strip().lower()
assert "t4" not in name and "v100" not in name, "T4/V100 cannot run this kernel (FP8 + VRAM). Switch runtime to A100."
import re
gb = float(re.search(r"(\d+)\s*MiB", out.stdout).group(1)) / 1024
assert gb >= 38, f"Need >=40GB VRAM for 27B FP8 weights + KV cache, found {gb:.0f}GB"
GPU_TYPE = "h100" if "h100" in name else "a100"
print("GPU ok:", GPU_TYPE)

In [ ]:
# Cell 3 — Kaggle credentials for takumuhata
# Upload credentials.json when the widget appears, OR pre-place it at /content/credentials.json
import json, os, stat
from pathlib import Path

dst = Path.home()/".kaggle"; dst.mkdir(exist_ok=True)
target = dst/"credentials.json"
if Path("/content/credentials.json").exists():
    target.write_bytes(Path("/content/credentials.json").read_bytes())
elif not target.exists():
    from google.colab import files
    up = files.upload()
    target.write_bytes(next(iter(up.values())))
os.chmod(target, 0o600)
who = json.loads(target.read_text()).get("username")
assert who == "takumuhata", f"expected credentials for takumuhata, got {who!r}"
print("kaggle credentials ok:", who)

In [ ]:
# Cell 4 — downloads (~30GB total, 10-25 min)
!pip -q install --upgrade kaggle papermill
from pathlib import Path
import json, subprocess, os

ROOT = Path("/content")
DS   = ROOT/"ds"; DS.mkdir(exist_ok=True)

def dl(cmd): print("+", " ".join(cmd), flush=True); subprocess.run(cmd, check=True)

# agent bundle (our patched v2: cap6144 + length-salvage + retry-shrink + time-awareness)
dl(["kaggle","datasets","download","takumuhata/taaf-anim-27b-patched","-p",str(DS/"bundle"),"--unzip","-q"])
# vLLM H100 wheelhouse (vllm 0.19.0 / torch 2.10.0 / flashinfer 0.6.6)
dl(["kaggle","datasets","download","driessmit1/arc3-vllm-h100-wheelhouse-v3","-p",str(DS/"wheelhouse"),"--unzip","-q"])
# competition data -> environment_files (25 public games)
dl(["kaggle","competitions","download","-c","arc-prize-2026-arc-agi-3","-p","/content/comp","-q"])
subprocess.run(["unzip","-q","-o","/content/comp/arc-prize-2026-arc-agi-3.zip","-d","/content/comp"],check=True)
# private FP8 model (~24GB, 16 shards)
dl(["kaggle","models","instances","versions","download",
    "foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1","-p","/content/model","--untar","-q"])
# the kernel notebook itself (run verbatim via papermill)
dl(["kaggle","kernels","pull","takumuhata/arc3-duck-qwen3-8-27b","-p","/content/kernel"])

# normalize model dir: files may land flat or one level deep after --untar
import glob
cand = [d for d in [Path("/content/model")]+[Path(p).parent for p in glob.glob("/content/model/**/config.json",recursive=True)]
        if (d/"config.json").exists() and list(d.glob("*.safetensors"))]
assert cand, "model download incomplete (no config.json + safetensors found)"
if cand[0] != Path("/content/model"):
    Path("/content/model_flat").mkdir(exist_ok=True)
    import shutil
    for f in cand[0].iterdir(): shutil.move(str(f), "/content/model_flat/")
    # repoint: keep /content/model -> flat for the links in cell 5
    import os
    for f in Path("/content/model_flat").iterdir():
        dst = Path("/content/model")/f.name
        if not dst.exists(): dst.symlink_to(f)
assert list((Path("/content/comp")/"environment_files").iterdir()), "environment_files missing"
assert (Path("/content/comp")/"arc_agi_3_wheels").is_dir(), "arc_agi_3_wheels missing from competition data"
print("downloads complete")

In [ ]:
# Cell 5 — /kaggle layout + env + vLLM site-packages (wheelhouse -> PyPI fallback)
import json, os, shutil, subprocess, sys
from pathlib import Path

WORK = Path("/content/working"); WORK.mkdir(exist_ok=True)
SITE = WORK/"vllm-site-packages"

# --- recreate the /kaggle mount tree the kernel hard-codes ---
links = {
    "/kaggle/input/datasets/takumuhata/taaf-anim-27b-patched":               "/content/ds/bundle",
    "/kaggle/input/taaf-anim-27b-patched":                                  "/content/ds/bundle",
    "/kaggle/input/datasets/driessmit1/arc3-vllm-h100-wheelhouse-v3":       "/content/ds/wheelhouse",
    "/kaggle/input/qwen3-8-27b-fp8-repacked-v1":                            "/content/model",
    "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1": "/content/model",
    "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files":"/content/comp/environment_files",
    "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels":  "/content/comp/arc_agi_3_wheels",
}
for lnk, tgt in links.items():
    p = Path(lnk); p.parent.mkdir(parents=True, exist_ok=True)
    if p.is_symlink() or p.exists(): p.unlink() if p.is_symlink() or p.is_file() else shutil.rmtree(p)
    p.symlink_to(tgt)
print("/kaggle tree:", *links.keys(), sep="\n  ")

# --- env the kernel cells read ---
env = {
    "TAAF_KAGGLE_WORKING_DIR": str(WORK),
    "TAAF_KAGGLE_BUNDLE_DIR":  "/content/ds/bundle",
    "TAAF_KAGGLE_SETUP_ENV":   str(WORK/"taaf_setup_env.json"),
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps({
        "takumuhata/taaf-anim-27b-patched": "/content/ds/bundle",
        "driessmit1/arc3-vllm-h100-wheelhouse-v3": "/content/ds/wheelhouse",
        "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot": "/content/model",
        "foysalemonshanto/qwen3-8-27b-fp8-repacked-v1": "/content/model"}),
    "KAGGLE_GPU_TYPE":  GPU_TYPE,          # 'a100' passes the substring check on 'A100-SXM4-40GB'
    "KAGGLE_GPU_COUNT": "1",
    "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
}
os.environ.update(env)
Path("/content/colab_env.json").write_text(json.dumps(env, indent=2))

# --- vLLM into SITE (same --target layout the setup script expects) ---
STAMP = SITE/".arc3-vllm-h100-wheelhouse-v3"
STAMP_TXT = "vllm==0.19.0 torch==2.10.0 flashinfer==0.6.6\n"
def site_ok():
    r = subprocess.run([sys.executable,"-c","import vllm, torch; print(vllm.__version__, torch.__version__, torch.cuda.get_arch_list())"],
                       env={**os.environ,"PYTHONPATH":str(SITE)},capture_output=True,text=True)
    print(r.stdout.strip() or r.stderr[-2000:])
    if r.returncode: return False
    import re
    al = re.findall(r"sm_(\d+)", r.stdout)
    return not al or "80" in al or GPU_TYPE=="h100"   # sm80 covers A100
def pip_install_site(extra):
    SITE.mkdir(parents=True, exist_ok=True)
    subprocess.run([sys.executable,"-m","pip","install","--target",str(SITE),
                    "--upgrade","--ignore-installed","--only-binary",":all:","--no-compile","--no-warn-conflicts"]+extra,check=True)

req = Path("/content/ds/wheelhouse/requirements.lock")
if req.exists():
    try:
        pip_install_site(["--no-index","--find-links","/content/ds/wheelhouse","--requirement",str(req)])
    except Exception as e: print("wheelhouse install failed:", e)
if not site_ok():
    print("wheelhouse unusable on this arch -> PyPI fallback")
    shutil.rmtree(SITE, ignore_errors=True)
    pip_install_site(["vllm==0.19.0","torch==2.10.0","flashinfer-python==0.6.6"])
    assert site_ok(), "vLLM install unusable on this GPU"
STAMP.write_text(STAMP_TXT)   # makes setup_commands skip its own install

# runtime deps of the taaf/benchmark sources (kernel pth adds src roots to sys.path)
# arc-agi/arcengine are competition wheels (not on PyPI); rest resolve from PyPI.
subprocess.run([sys.executable,"-m","pip","install","-q",
                "--find-links","/content/comp/arc_agi_3_wheels",
                "arc-agi","arcengine","imageio","imageio-ffmpeg","scipy","pandas",
                "matplotlib","openai","httpx","pillow","python-dotenv","requests"],shell=False)
import importlib; importlib.import_module("arc_agi"); importlib.import_module("arcengine")
print("site-packages ready")

In [ ]:
# Cell 6 — run the Kaggle kernel notebook verbatim (5-8h on A100)
import os, subprocess, sys
from pathlib import Path
Path("/content/working").mkdir(exist_ok=True)
os.environ.update(__import__("json").loads(Path("/content/colab_env.json").read_text()))
r = subprocess.run([sys.executable,"-m","papermill","/content/kernel/arc3-duck-qwen3-8-27b.ipynb",
                    "/content/out.ipynb","--cwd","/content/working","--log-output"],cwd="/content/working")
print("papermill exit:", r.returncode)
if r.returncode:
    print("--- vllm server log tail ---")
    log = Path("/content/working/vllm-openai-server.log")
    if log.exists(): print("\n".join(log.read_text(errors="replace").splitlines()[-60:]))

### Throughput tuning (learned from run 1 on A100-40GB)

Run 1 finished but scored 0.11: every game died on ~875-900s request timeouts.
On a single A100-40GB the KV cache holds ~4 concurrent 32k-context requests; the
kernel's default `concurrency=28` queues 20+ requests per turn, so analyzer
requests starve and every game burns its wall-clock budget in the queue.

The next cell patches the pulled kernel to `concurrency=6` and replaces the
bundle's `tool_agent.py` with the patched copy in dataset `taaf-anim-27b-patched`
(which adds a request-budget-aware `max_tokens` clamp so a slow turn returns a
truncated-but-usable response instead of timing out). Skip it if you want the
verbatim kernel config.

In [ ]:
# Cell 6a - throughput tuning: lower game concurrency + patched tool_agent
import json, glob, shutil

KERNEL = glob.glob('/content/kernel/*.ipynb')[0]
nb = json.load(open(KERNEL))
hits = 0
for cell in nb['cells']:
    if cell.get('cell_type') != 'code':
        continue
    src = ''.join(cell.get('source', []))
    if 'concurrency = 28' in src:
        src = src.replace('concurrency = 28', 'concurrency = 6')
        lines = src.split('\n')
        cell['source'] = [l + '\n' for l in lines[:-1]] + [lines[-1]]
        hits += 1
json.dump(nb, open(KERNEL, 'w'))
print('concurrency=6 patched in', hits, 'cells')

!mkdir -p /content/ds_latest
!kaggle datasets download -d takumuhata/taaf-anim-27b-patched -p /content/ds_latest --unzip -q
import glob as g
src = g.glob('/content/ds_latest/src/*/inference/agent/tool_agent.py')
dst = g.glob('/content/ds/bundle/src/*/inference/agent/tool_agent.py')
assert src and dst, (src, dst)
shutil.copyfile(src[0], dst[0])
print('tool_agent replaced:', src[0], '->', dst[0])

In [ ]:
# Cell 7 — results
import json
from pathlib import Path
w = Path("/content/working")
for f in ("score.json","summary.txt","benchmark.json"):
    p = w/f
    if p.exists():
        print(f"=== {f} ==="); print(p.read_text()[:4000])
sub = list(w.glob("submission*.parquet"))
print("submission:", sub)

In [ ]:
# Cell 8 (optional) — fallback retry if cell 6 OOMed / vLLM failed to start
# Patches the bundle's setup to halve context and adds enforce-eager, then re-runs.
import json, re, os, sys, subprocess
from pathlib import Path
os.environ.update(json.loads(Path("/content/colab_env.json").read_text()))
cmd_file = Path("/content/ds/bundle/setup_commands.json")
cmds = json.loads(cmd_file.read_text())
new = []
for cmd in cmds:
    cmd, n1 = re.subn(r"(?m)^VLLM_MAX_MODEL_LEN\s*=\s*\d+\s*$", "VLLM_MAX_MODEL_LEN = 32768", cmd)
    cmd = cmd.replace("'--max-model-len',", "'--enforce-eager',\\n        '--max-model-len',")
    new.append(cmd); print("patched:", n1)
cmd_file.write_text(json.dumps(new))
r = subprocess.run([sys.executable,"-m","papermill","/content/kernel/arc3-duck-qwen3-8-27b.ipynb",
                    "/content/out2.ipynb","--cwd","/content/working","--log-output"],cwd="/content/working")
print("retry exit:", r.returncode)